# Módulo de Configuración Histórica
## Notebook 02 — Asignaciones Versionadas & Detalle de Esquemas

Este notebook crea las cuatro tablas que contienen el historial versionado:

| Capa | Tabla | Descripción |
|---|---|---|
| Asignaciones | `usuario_rol_historico` | Usuario ↔ Rol con vigencia temporal |
| Asignaciones | `rol_esquema_historico` | Rol ↔ Esquema de compensación con vigencia |
| Detalle | `esquema_evaluacion_historico` | Evaluaciones del esquema con su peso (%) |
| Detalle | `meta_evaluacion_historico` | Meta por evaluación, esquema, año y mes |

### Cadena de relaciones
```
dim_usuario
    └─► usuario_rol_historico  (fecha_inicio / fecha_fin)
            └─► dim_rol
                    └─► rol_esquema_historico  (fecha_inicio / fecha_fin)
                                └─► esquema_evaluacion_historico  (peso)
                                            └─► dim_evaluacion
                                            └─► meta_evaluacion_historico
                                                        └─► anio + mes  (período directo)
```

> **Prerrequisito**: Ejecuta primero **01_dimensiones_config_historico.ipynb** y asegúrate de que el Lakehouse adjunto sea el mismo.

---
## 0 · Configuración del Lakehouse

In [ ]:
# ─── Debe coincidir con el valor usado en el Notebook 01 ─────────────────────
LAKEHOUSE_NAME = "BI - Bandelta LH"
# ─────────────────────────────────────────────────────────────────────────────

spark.sql(f"USE `{LAKEHOUSE_NAME}`")
print(f"✅ Usando Lakehouse: {LAKEHOUSE_NAME}")

---
## 1 · usuario_rol_historico

**Capa de Asignaciones** — registra qué rol tuvo cada usuario y en qué período.

- `fecha_fin = NULL` significa que la asignación sigue **vigente**.
- Un usuario puede tener múltiples registros si cambió de rol.
- La clave de negocio para consultar el rol activo es:
  `WHERE fecha_inicio <= :fecha AND (fecha_fin IS NULL OR fecha_fin >= :fecha)`

In [ ]:
spark.sql("""
CREATE TABLE IF NOT EXISTS usuario_rol_historico (

    -- Identidad
    id_asignacion      INT        NOT NULL  COMMENT 'Llave primaria de la asignación',

    -- Relaciones (FK lógicas — Delta no impone FK físicas)
    id_usuario         INT        NOT NULL  COMMENT 'FK → dim_usuario.id_usuario',
    id_rol             INT        NOT NULL  COMMENT 'FK → dim_rol.id_rol',

    -- Vigencia temporal
    fecha_inicio       DATE       NOT NULL  COMMENT 'Fecha desde la que aplica este rol',
    fecha_fin          DATE                 COMMENT 'Fecha hasta la que aplica; NULL = vigente',

    -- Metadata operativa
    activo             BOOLEAN    NOT NULL  COMMENT 'Bandera de registro activo',
    fecha_creacion     TIMESTAMP  NOT NULL  COMMENT 'Cuándo se registró la asignación',
    usuario_creacion   STRING               COMMENT 'Quién realizó la asignación'
)
USING DELTA
COMMENT 'Historial versionado de roles por usuario'
TBLPROPERTIES (
    'delta.minReaderVersion' = '1',
    'delta.minWriterVersion' = '2'
)
""")

print("✅ Tabla usuario_rol_historico creada (o ya existía).")

In [ ]:
spark.sql("DESCRIBE TABLE usuario_rol_historico").show(truncate=False)

---
## 2 · rol_esquema_historico

**Capa de Asignaciones** — vincula un rol con su esquema de compensación y registra los cambios en el tiempo.

Permite que el mismo rol tenga **reglas distintas** en distintos momentos  
(p.ej. el rol "Asesor Comercial" tenía en 2023 un esquema diferente al de 2024).

In [ ]:
spark.sql("""
CREATE TABLE IF NOT EXISTS rol_esquema_historico (

    -- Identidad
    id_asignacion_esquema  INT        NOT NULL  COMMENT 'Llave primaria de la asignación',

    -- Relaciones
    id_rol                 INT        NOT NULL  COMMENT 'FK → dim_rol.id_rol',
    id_esquema             INT        NOT NULL  COMMENT 'Identificador del esquema de compensación',
    nombre_esquema         STRING     NOT NULL  COMMENT 'Nombre descriptivo del esquema',

    -- Vigencia temporal
    fecha_inicio           DATE       NOT NULL  COMMENT 'Fecha desde la que aplica el esquema',
    fecha_fin              DATE                 COMMENT 'Fecha hasta la que aplica; NULL = vigente',

    -- Metadata operativa
    activo                 BOOLEAN    NOT NULL  COMMENT 'Bandera de registro activo',
    fecha_creacion         TIMESTAMP  NOT NULL  COMMENT 'Cuándo se registró la asignación',
    usuario_creacion       STRING               COMMENT 'Quién configuró el esquema'
)
USING DELTA
COMMENT 'Historial versionado del esquema de compensación por rol'
TBLPROPERTIES (
    'delta.minReaderVersion' = '1',
    'delta.minWriterVersion' = '2'
)
""")

print("✅ Tabla rol_esquema_historico creada (o ya existía).")

In [ ]:
spark.sql("DESCRIBE TABLE rol_esquema_historico").show(truncate=False)

---
## 3 · esquema_evaluacion_historico

**Capa de Detalle** — desglosa qué evaluaciones componen cada esquema y con qué peso porcentual.

Regla de negocio: **la suma de `peso` de todas las evaluaciones activas de un esquema debe ser igual a 1.0 (100%)**.

In [ ]:
spark.sql("""
CREATE TABLE IF NOT EXISTS esquema_evaluacion_historico (

    -- Identidad
    id_detalle      INT            NOT NULL  COMMENT 'Llave primaria del detalle',

    -- Relaciones
    id_esquema      INT            NOT NULL  COMMENT 'FK → rol_esquema_historico.id_esquema',
    id_evaluacion   INT            NOT NULL  COMMENT 'FK → dim_evaluacion.id_evaluacion',

    -- Reglas de compensación
    peso            DECIMAL(5, 4)  NOT NULL  COMMENT 'Peso relativo p.ej. 0.3000 = 30%; la suma por esquema debe ser 1.0',

    -- Vigencia temporal
    fecha_inicio    DATE           NOT NULL  COMMENT 'Fecha desde la que aplica este peso',
    fecha_fin       DATE                     COMMENT 'Fecha hasta la que aplica; NULL = vigente',

    -- Metadata operativa
    activo          BOOLEAN        NOT NULL  COMMENT 'Bandera de registro activo',
    fecha_creacion  TIMESTAMP      NOT NULL  COMMENT 'Cuándo se configuró este peso'
)
USING DELTA
COMMENT 'Detalle versionado de evaluaciones y pesos por esquema de compensación'
TBLPROPERTIES (
    'delta.minReaderVersion' = '1',
    'delta.minWriterVersion' = '2'
)
""")

print("✅ Tabla esquema_evaluacion_historico creada (o ya existía).")

In [ ]:
spark.sql("DESCRIBE TABLE esquema_evaluacion_historico").show(truncate=False)

---
## 4 · meta_evaluacion_historico

**Capa de Detalle** — define el **valor objetivo** que debe alcanzar cada evaluación en cada período.

Esta es la tabla más granular: cruza esquema × evaluación × período.  
Permite responder: _"¿cuántas ventas debía hacer el Asesor Comercial en marzo 2024?"_

In [ ]:
spark.sql("""
CREATE TABLE IF NOT EXISTS meta_evaluacion_historico (

    -- Identidad
    id_meta          INT            NOT NULL  COMMENT 'Llave primaria de la meta',

    -- Relaciones (clave compuesta de negocio: esquema + evaluación + período)
    id_esquema       INT            NOT NULL  COMMENT 'FK → rol_esquema_historico.id_esquema',
    id_evaluacion    INT            NOT NULL  COMMENT 'FK → dim_evaluacion.id_evaluacion',

    -- Período directo (sin tabla catálogo)
    anio             INT            NOT NULL  COMMENT 'Año del período, p.ej. 2024',
    mes              INT            NOT NULL  COMMENT 'Mes del período 1-12',

    -- Valor de la meta
    valor_meta       DECIMAL(18, 4) NOT NULL  COMMENT 'Valor objetivo de la evaluación en el período',
    valor_minimo     DECIMAL(18, 4)           COMMENT 'Umbral mínimo para considerar cumplimiento parcial',
    valor_maximo     DECIMAL(18, 4)           COMMENT 'Techo de referencia (opcional)',

    -- Metadata operativa
    fecha_creacion   TIMESTAMP      NOT NULL  COMMENT 'Cuándo se registró la meta',
    usuario_creacion STRING                   COMMENT 'Quién cargó la meta'
)
USING DELTA
COMMENT 'Meta objetivo por evaluación, esquema, año y mes'
TBLPROPERTIES (
    'delta.minReaderVersion' = '1',
    'delta.minWriterVersion' = '2'
)
""")

print("✅ Tabla meta_evaluacion_historico creada (o ya existía).")

In [ ]:
spark.sql("DESCRIBE TABLE meta_evaluacion_historico").show(truncate=False)

---
## 5 · Vista de relaciones: verificación del modelo completo

In [ ]:
# Lista todas las tablas del Lakehouse para confirmar que el modelo está completo
tablas = spark.sql("SHOW TABLES").toPandas()
display(tablas)

# Resumen de conteo
todas = [
    "dim_usuario", "dim_rol", "dim_evaluacion",
    "usuario_rol_historico", "rol_esquema_historico",
    "esquema_evaluacion_historico", "meta_evaluacion_historico"
]

print(f"\n{'TABLA':<35} {'REGISTROS':>10}")
print("-" * 47)
for t in todas:
    cnt = spark.sql(f"SELECT COUNT(*) as c FROM {t}").collect()[0].c
    print(f"{t:<35} {cnt:>10,}")

print("\n✅ Notebook 02 completado — modelo de Configuración Histórica listo.")